# Análisis Geoespacial IPM - Zona de Intervención: Roosevelt

Este notebook prioriza la visualización cartográfica de las 5 variables más relevantes del Índice de Pobreza Multidimensional (IPM) en el corredor Roosevelt (Buffer 100m).

### Contexto y Cifras Generales (Corte Mayo 2026)

Para la interpretación de los resultados, se deben considerar dos lógicas opuestas:
1. **Índice de Condición Social (ICS):** Es un indicador directo; un valor del **100% representa el escenario óptimo** (bienestar y acceso pleno).
2. **Índice de Pobreza Multidimensional (IPM):** Es un indicador inverso; un valor del **100% representa el escenario crítico** (pobreza absoluta).

A nivel distrital, Cali presenta una marcada brecha territorial. La **severidad de la pobreza (IPM promedio en manzanas con incidencia > 0)** es del **15.28% en el área urbana**, mientras que en el **área rural se dispara al 38.33%**, impulsada principalmente por carencias en infraestructura básica en los corregimientos.

Este análisis se enfoca en el corredor Roosevelt para identificar las privaciones específicas que afectan a este territorio de intervención.

In [ ]:
# 1. Instalación de dependencias (solo en Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas matplotlib seaborn openpyxl -q

In [ ]:
# 2. Importar librerías
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from matplotlib.patheffects import withStroke
from matplotlib.colors import BoundaryNorm
import matplotlib.patches as mpatches

sns.set_style('white')
print('Librerías listas')

In [ ]:
# 3. Definir rutas con detección de entorno
REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !cd {REPO_DIR} && git pull
    BASE_DIR = REPO_DIR
else:
    # Local: Intentar detectar carpeta indice_Pobreza
    if os.path.exists('indice_Pobreza'): BASE_DIR = '.'
    elif os.path.exists('../indice_Pobreza'): BASE_DIR = '..'
    elif os.path.exists('../../indice_Pobreza'): BASE_DIR = '../..'
    else: BASE_DIR = '.'

DATA_BASE = os.path.join(BASE_DIR, 'indice_Pobreza/data')

# Capas
PATH_IPM_VARS_FULL = os.path.join(DATA_BASE, 'geojson_ipm/Mzn_ipm_variables.geojson')
PATH_MANZANAS_FONDO = os.path.join(DATA_BASE, 'geojson_Manzanas_catastrales/geojson_Manzanas_catastrales.geojson')
PATH_AREA_ESTUDIO = os.path.join(DATA_BASE, 'geojson_poligonos_territorio_ITT/poligono_Roosevelt_Buffer_100.geojson')

# Cargar datos
gdf_full = gpd.read_file(PATH_IPM_VARS_FULL)
gdf_fondo = gpd.read_file(PATH_MANZANAS_FONDO)
gdf_area = gpd.read_file(PATH_AREA_ESTUDIO)

# Unificar CRS a WGS84
gdf_full = gdf_full.to_crs('EPSG:4326')
gdf_fondo = gdf_fondo.to_crs('EPSG:4326')
gdf_area = gdf_area.to_crs('EPSG:4326')

# Filtrado espacial preciso (Manzanas dentro del Buffer)
gdf_p = gdf_full.to_crs("EPSG:3115")
area_p = gdf_area.to_crs("EPSG:3115")
idx = gpd.sjoin(gdf_p.copy().assign(geometry=gdf_p.centroid), area_p[['geometry']], how='inner', predicate='within').index
gdf = gdf_full.loc[idx].copy()

print(f'Manzanas detectadas en Roosevelt: {len(gdf)}')

In [ ]:
# 4. Diccionario Estandarizado y Top 5
COLS_MAP = {
    'ANALF_': 'Analfabetismo', 'BAJO_': 'Bajo logro educativo', 
    'INFANCIA_': 'Barreras primera infancia', 'INASIS_': 'Inasistencia escolar', 
    'REZAGO_': 'Rezago escolar', 'TRAB_INFAN': 'Trabajo infantil', 
    'DEPEN_': 'Dependencia económica', 'INFOR_': 'Informalidad', 
    'SALUD_': 'Barreras de salud', 'ASEGU_': 'Sin aseguramiento en salud', 
    'HACI_': 'Hacinamiento crítico', 'PARED_': 'Paredes precarias', 
    'EXCRE_': 'Eliminación inadecuada de excretas', 'PISOS_': 'Pisos precarios', 
    'AGUA_': 'Sin acceso a agua mejorada'
}

top_5 = gdf[list(COLS_MAP.keys())].mean().sort_values(ascending=False).head(5).index.tolist()
print('Top 5 variables críticas en Roosevelt:')
for v in top_5: print(f'- {COLS_MAP[v]}')

In [ ]:
# 5. Función de mapeo con Efecto Atlas, Contraste y Rangos Dinámicos
def plot_ipm_variable_advanced(gdf_zone, gdf_back, gdf_poly, column, title):
    fig, ax = plt.subplots(1, 1, figsize=(20, 14), facecolor='white')
    
    # Extensión dinámica
    b = gdf_poly.total_bounds
    px, py = (b[2]-b[0])*0.25, (b[3]-b[1])*0.25
    extent = [b[0]-px, b[1]-py, b[2]+px, b[3]+py]
    
    # Rangos dinámicos por variable
    vals = gdf_zone[column].dropna()
    if vals.max() > 0:
        breaks = sorted(list(set([0, 0.1] + list(np.linspace(vals[vals>0].min() if not vals[vals>0].empty else 1, vals.max(), 4)))))
    else:
        breaks = [0, 1, 2, 3, 4, 5]
    
    cmap = plt.get_cmap('viridis', len(breaks)-1)
    norm = BoundaryNorm(breaks, cmap.N)
    
    # Ploteo de capas
    gdf_back.plot(ax=ax, facecolor='#F2F2F2', edgecolor='#D9D9D9', linewidth=0.5, zorder=1)
    gdf_poly.plot(ax=ax, facecolor='none', edgecolor='#E63946', linewidth=3, linestyle='--', alpha=0.5, zorder=2)
    gdf_zone.plot(column=column, cmap=cmap, norm=norm, edgecolor='white', linewidth=1, ax=ax, zorder=3, alpha=0.9)
    
    # Etiquetas de Contraste
    c = gdf_zone.to_crs("EPSG:3115").geometry.centroid.to_crs(gdf_zone.crs)
    for x, y, label in zip(c.x, c.y, gdf_zone[column]):
        color = 'white' if label > (vals.max()*0.6) else 'black'
        ax.annotate(f'{label:.1f}%', xy=(x,y), ha='center', va='center', fontsize=10, fontweight='black', color=color,
                    path_effects=[withStroke(linewidth=3, foreground='white' if color=='black' else 'black', alpha=0.8)], zorder=5)
    
    # Leyenda Estilizada
    lp = [mpatches.Patch(facecolor=cmap(i), edgecolor='gray', label=f"{breaks[i]:.1f}% - {breaks[i+1]:.1f}%") for i in range(len(breaks)-1)]
    lp.append(mpatches.Patch(facecolor='none', edgecolor='#E63946', linewidth=2, linestyle='--', label='Buffer Roosevelt (100m)'))
    ax.legend(handles=lp, title=f"Privación: {title}", loc='upper left', bbox_to_anchor=(1, 1), fontsize=11, frameon=True)
    
    ax.set_title(f"Distribución Espacial: {title.upper()}\nZona de Intervención ITT Roosevelt", fontsize=18, fontweight='bold', pad=25)
    ax.set_xlim(extent[0], extent[2]); ax.set_ylim(extent[1], extent[3])
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
# 6. Generación de Cartografía Crítica
for var in top_5:
    plot_ipm_variable_advanced(gdf, gdf_fondo, gdf_area, var, COLS_MAP[var])

# Guardar resultado para descarga
PATH_OUT = 'Mzn_ipm_variables_Roosevelt.geojson'
gdf.to_file(PATH_OUT, driver='GeoJSON')

## 7. Validación de Estándares Técnicos
Este notebook cumple con las directrices de geo-informática de Santiago de Cali (Mayo 2026):
1. **Accesibilidad:** Uso de la paleta **Viridis** para IPM.
2. **Efecto Atlas:** Manzanas de contexto y fondo completo.
3. **Contraste:** Etiquetas inteligentes (Blanco/Negro) según el fondo.
4. **Precisión:** Datos ajustados a **1 decimal**.

In [ ]:
# 8. Descargar GeoJSON (Colab)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(PATH_OUT)
    print(f'Descarga: {PATH_OUT}')
else:
    print(f'Archivo disponible en: {os.path.abspath(PATH_OUT)}')